In [ ]:
!pip install squidpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import tarfile
tar_path = '/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/Croped_Images_3.tar'

with tarfile.open(tar_path) as tar:
    tar.extractall('/content/Extracted/')

/tmp/ipykernel_1387/2788671346.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/Extracted/')


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scanpy as sc

import os
import sys
os.chdir('/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Train/Modified_Steamboat_CNN')
sys.path.append('../../')

In [2]:
import Modified_Steamboat_CNN as MSC
import Modified_Steamboat_CNN.dataset as MSC_dataset
import Modified_Steamboat_CNN.model as MSC_model
import importlib

importlib.reload(MSC_model)
importlib.reload(MSC_dataset)
importlib.reload(MSC)

<module 'Modified_Steamboat_CNN' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Train/Modified_Steamboat_CNN/../../Modified_Steamboat_CNN/__init__.py'>

## Dataset

In [3]:
ad = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI_adata_r.h5ad")

adata = ad[:1680].copy()

In [4]:
adata = MSC.prep_adatas(adata, norm=True, log1p=True)

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Train/Modified_Steamboat_CNN/../../Modified_Steamboat_CNN/dataset.py:135: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(adata)


INFO     Creating graph using `generic` coordinates and `None` transform and `1` libraries.                        


In [5]:
img_dir = '/content/Extracted/Croped_Images_3'
dataset = MSC.make_dataset(adata, image_dir = img_dir, image_ext ='.png', sparse_graph=True)

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Train/Modified_Steamboat_CNN/../../Modified_Steamboat_CNN/dataset.py:259: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  cell['id'] = adata.obs['cell_id'][i]


Building dataset


Preloading images: 100%|██████████| 1680/1680 [00:17<00:00, 96.85it/s] 


## Train

In [6]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset=dataset,
    batch_size=512,
    shuffle=False,
)

In [7]:
for sample in dataloader:

  x_global = sample['X_global'][0]
  x_local = sample['X_local']
  m_global = sample['M_global']
  m_local = sample['M_local']
  image = sample['image']
  cell_id = sample['cell_id']
  adj = sample['adj']

  print(x_global.shape)
  print(x_local.shape)
  print(m_global.shape)
  print(m_local.shape)
  print(image.shape)
  print(x_global[adj].shape)
  break

torch.Size([1680, 313])
torch.Size([524, 313])
torch.Size([524, 1680, 500])
torch.Size([524, 500])
torch.Size([524, 3, 64, 64])
torch.Size([524, 8, 313])


model training

In [7]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device)

cuda


In [8]:
model = MSC.model.Steamboat(
    features=len(adata.var_names.tolist()),
    image_size=dataset[0]['image'].shape,
    morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1],
    n_heads=32
    )
model = model.to(device)


In [ ]:
loss = model.fit(
    dataloader,
    entry_masking_rate=1.0,
    device=device,
    max_epoch=10000,
    loss_fun=torch.nn.MSELoss(reduction='mean'),
    opt=torch.optim.Adam,
    opt_args=dict(lr=0.01),
    stop_eps=1e-7, report_per=200, stop_tol=200,
    return_loss=True,
    )

Epoch 1: loss =  1.55237
